# Design bit-source modules

This notebook develops and tests DSP modules that generate sequences of bits to be used as input for digital communication simulations.

The sources implemented below include a random bit generator and a source that converts text strings into sequences of bits.

Two design approaches are considered:

* a functional design: functionality is provided by functions
* an object-oriented design: a class-based approach where functionality is encapsulated within classes


Throughout, the preferred documentation format using the [NumPy documentation conventions](https://numpydoc.readthedocs.io/en/latest/format.html) is followed.

Note also the practice to provide a brief unit test for the function.

In [1]:
## import standard modules
import numpy as np

## Functional design

In the functional design approach, we implement bit-source modules as functions. Each function performs a specific task, such as generating random bits or converting text to a bit sequence. This approach emphasizes simplicity and ease of use.

### Random bit source

The random bit source generates a sequence of bits (0s and 1s) with equal probability. This is useful for simulating random digital data in communication systems.

The implementation is trivial since it can leverage the `np.random.randint` function from the NumPy library to generate random integers (0 or 1) for the bit sequence.

In [2]:
def random_bit_source(num_bits: int) -> np.ndarray:
    """Generate a sequence of random bits (0s and 1s) with equal probability.

    Parameters:
    -----------
    num_bits (int): 
        The number of bits to generate.

    Returns:
    --------
    np.ndarray: 
        An array of random bits (0s and 1s) of length `num_bits`.

    Example:
    --------
    >>> random_bit_source(5)
    array([0, 1, 0, 1, 1])
    
    """
    rng = np.random.default_rng()
    return rng.integers(low=0, high=2, size=num_bits, dtype=np.uint8)

In [3]:
def test_random_bit_source():
    """Unit test for the random_bit_source function."""

    num_bits = 10
    
    bits = random_bit_source(num_bits)
    assert isinstance(bits, np.ndarray), "Output should be a numpy array"
    assert bits.shape[0] == num_bits, "Output array should have the correct length"
    assert np.all(np.isin(bits, [0, 1])), "All elements should be 0 or 1"

test_random_bit_source()

### Text string source

The text string source converts a Unicode text string into a sequence of bits.

Before defining the function, a helper function that converts a byte to a sequence of 8 bits (with MSB first)is defined.

In [4]:
def byte_to_bits(b: int) -> np.ndarray:
    """convert a single byte to a sequence of 8 bits (MSB first)

    Parameters:
    ----------
    b (int):
        a single byte (0-255)

    Returns:
    --------
    np.ndarray:
        a NumPy vector of bits, stored as uint8

    Example:
    --------
    >>> byte_to_bits(5)
    array([0, 0, 0, 0, 0, 1, 0, 1], dtype=uint8)
    """

    # allocate memory for bits
    bits = np.zeros(8, dtype=np.uint8)

    # define the mask
    mask = 128

    for n in range(8):
        # extract the MSB and store it
        bits[n] = (b & mask) >> 7
        # shift the bits by one position
        b = b << 1

    return bits

With this helper, converting a text string into a sequence of bits becomes straightforward.

In [5]:
def string_source(string: str) -> np.ndarray:
    """convert a string to a vector of bits

    Parameters:
    -----------
    string (str): 
        The string to be converted into a bit-sequence; maybe ASCII or UTF-8

    Returns:
    --------
    np.ndarray: 
        A numpy array representing the bit sequence of the input string.

    Example:
    --------
    >>> string_source("ABC")
    array([0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1])
    """
    # convert a string to a sequence of bytes
    bb = string.encode()  # convert the Unicode string to a sequence of bytes
    Nb = len(bb)

    # allocate space
    bits = np.zeros(8 * Nb, dtype=np.uint8)

    for n in range(Nb):
        bits[8 * n : 8 * (n + 1)] = byte_to_bits(bb[n])

    return bits


In [6]:
## unit tests for string_source
def test_string_source():
    text = "A"
    expected_bits = np.array([0, 1, 0, 0, 0, 0, 0, 1], dtype=np.uint8)
    assert np.array_equal(string_source(text), expected_bits), "Test failed for input 'A'"

    text = "Äö"
    expected_bits = np.array([
        1, 1, 0, 0, 0, 0, 1, 1,  # Ä (U+00C4) in UTF-8: 0xC3 0x84
        1, 0, 0, 0, 0, 1, 0, 0,
        1, 1, 0, 0, 0, 0, 1, 1,
        1, 0, 1, 1, 0, 1, 1, 0,   # ö (U+00F6) in UTF-8: 0xC3 0xB6
    ], dtype=np.uint8)
    assert np.array_equal(string_source(text), expected_bits), "Test failed for input 'Äö'"

test_string_source()

## Class-based Design for Bit Sources

In this section, we develop bit sources using a class-based approach. Each bit source is encapsulated within a class, providing methods to generate and retrieve sequences of bits. 

It may not be evident for this application, but the ability to maintain state inside an object often simplifies the management of complex behaviors and interactions in more advanced digital communication systems.

We begin by defining a base class for bit sources, which will provide a common interface for all specific bit source implementations by defining the function(s) that each specific bit source must provide.

In [7]:
class BitSource:
    """Base class for bit sources.
    
    Methods:
    --------
    get_bits()
        Retrieve the generated bits.
    """

    def __init__(self):
        pass

    def __repr__(self):
        return f"{self.__class__.__name__}()"

    def get_bits(self, n:int|None)->np.ndarray:
        """Retrieve the the next n bits.
        
        Parameters:
        -----------
        n : int, optional
            The number of bits to retrieve. If None, retrieve all available bits.

        Returns:
        --------
        np.ndarray
            Array of retrieved bits.
        """
        raise NotImplementedError("Subclasses must implement this method.")

### Class for generating random bits

The `RandomBitSource` class generates random bits on demand. It inherits from the `BitSource` base class and implements the `get_bits` method to produce a specified number of random bits.

In [8]:
class RandomBitSource(BitSource):
    """Class for generating random bits.
    
    Methods:
    --------
    get_bits()
        Retrieve the generated bits.
    """

    def __init__(self):
        super().__init__()

        # Initialize the random number generator
        self.rng = np.random.default_rng()

    def get_bits(self, n:int|None=None)->np.ndarray:
        """Retrieve the next n random bits.

        Parameters:
        -----------
        n : int, optional
            The number of bits to retrieve. If None, retrieve all available bits.

        Returns:
        --------
        np.ndarray
            Array of retrieved random bits.

        Example:
        --------
        >>> rbs = RandomBitSource()
        >>> rbs.get_bits(5)
        array([0, 1, 0, 1, 1])
        """
        if n is None:
            raise ValueError("Number of bits 'n' must be specified.")
        
        return self.rng.integers(0, 2, size=n)

In [9]:
## unit test

def test_random_bit_source():
    rbs = RandomBitSource()
    num_bits = 10
        
    bits = rbs.get_bits(num_bits)
    assert isinstance(bits, np.ndarray), "Output should be a numpy array"
    assert bits.shape[0] == num_bits, "Output array should have the correct length"
    assert np.all(np.isin(bits, [0, 1])), "All elements should be 0 or 1"

test_random_bit_source()

In [10]:
rbs = RandomBitSource()
rbs

RandomBitSource()

### Class for generating bits from a text string

The `TextBitSource` class generates bits based on the binary representation of a given text string.

It supports retrieving a specified number of bits from the binary representation of the text. It can be invoked repeatedly to retrieve bits in chunks of desired length. This is harder to do with a functional implementation.

In [11]:
class TextBitSource(BitSource):
    """Class for generating bits from a text string.
    
    Methods:
    --------
    get_bits()
        Retrieve the generated bits.
    """

    def __init__(self, text: str):
        super().__init__()
        self.text = text

        # internal state for tracking the bit sequence and current index
        self._bits = self._text_to_bits(text)
        self._index = 0

    def __repr__(self):
        return f"{self.__class__.__name__}(text={self.text!r})"

    def _text_to_bits(self, text: str) -> np.ndarray:
        # private helper method to convert text to bits
        bb = text.encode()  # convert the Unicode string to a sequence of bytes
        Nb = len(bb)
    
        # allocate space
        bits = np.zeros(8 * Nb, dtype=np.uint8)
    
        for n in range(Nb):
            bits[8 * n : 8 * (n + 1)] = byte_to_bits(bb[n])
        
        return bits

    def get_bits(self, n: int | None = None) -> np.ndarray:
        """Retrieve the next n bits from the text bit sequence.

        Parameters:
        -----------
        n : int, optional
            The number of bits to retrieve. If None, retrieve all remaining bits.

        Returns:
        --------
        np.ndarray
            Array of retrieved bits.

        Note:
        -----
        If the requested number of bits exceeds the remaining bits, only the available bits will be returned. When an empty array is returned, the source is exhausted and will not provide any more bits.

        Example:
        --------
        >>> tbs = TextBitSource("abc")
        >>> for _ in range(4):
        ...     print(tbs.get_bits(8))
        [0 1 1 0 0 0 0 1]
        [0 1 1 0 0 0 1 0]
        [0 1 1 0 0 0 1 1]
        []
        """
        if n is None:
            # if n is None, retrieve all remaining bits
            n = len(self._bits) - self._index

        if self._index + n > len(self._bits):
            # if the requested number of bits exceeds the remaining bits, adjust n
            # and return only the available bits
            n = len(self._bits) - self._index

        result = self._bits[self._index:self._index + n]
        self._index += n
        
        return result

In [12]:
## unit test for TextBitSource

def test_text_bit_source():
    tbs = TextBitSource("abc")
    for _ in range(3):
        assert tbs.get_bits(8).shape == (8,)
    # after consuming all bits, the next call should return an empty array
    assert tbs.get_bits(8).shape == (0,)

    # test retrieving all remaining bits
    tbs = TextBitSource("abc")
    all_bits = tbs.get_bits()
    assert all_bits.shape == (24,)
    # after consuming all bits, the next call should return an empty array
    assert tbs.get_bits().shape == (0,)

test_text_bit_source()

In [13]:
tbs = TextBitSource("abc")
tbs

TextBitSource(text='abc')

## Summary

We have implemented a set of functions and classes for generating bit sequences.

These functions and classes have been copied into the `sources` module (file: `src/dc_632_26/sources.py`) for reuse in other parts of the project.
The cell below verifies that the functions and classes can be imported from the `sources` module.

Similarly, the unit tests have been copied into a separate test module (file: `tests/test_sources.py`) to ensure the correctness of the implementations after moving them to the `sources` module.

In [14]:
# this should produce no output, especially no errors
from dc_632_26.sources import random_bit_source, string_source, RandomBitSource, TextBitSource